In [85]:
%matplotlib inline
from simulation import *
from math import *
from cmath import exp, phase
import numpy as np
import os
import matplotlib.pyplot as plt


golden_root_dir = "../resource/golden_data_latest/near_field/hyper_lith_kirchhoff/case1"
aerial = np.loadtxt(os.path.join(os.path.abspath(os.path.expanduser(golden_root_dir)), "thin_mask_2d_aerial_binary.txt"), delimiter='\t')
near_field = np.loadtxt(os.path.join(os.path.abspath(os.path.expanduser(golden_root_dir)), "thin_mask_2d_mag_binary.txt"), delimiter='\t')
pupil = np.loadtxt(os.path.join(os.path.abspath(os.path.expanduser(golden_root_dir)), "thin_mask_2d_pupil_intensity_binary.txt"), delimiter='\t')

In [ ]:
wavelength = 13.0
NA = 0.9
pitch = 16
cd = 8
dx = 1
size = int(ceil(pitch / dx))
center_y, center_x = size // 2, size // 2

grid_info_2d = grid_info_dbu_2d_s.create_grid_info_bloch_mode([size, size], wavelength, 0.0, NA, [[-pitch/2, -pitch/2], [pitch/2, pitch/2]], 1e-6)
print(grid_info_2d)

freq, _ = grid_info_2d.fourier.step


In [ ]:
background = 1#polar_to_complex(0.86065, 0)
absorber = 0#polar_to_complex(0.0968, -2.675)
g2 = geo_manager([[
        [int(-cd/2/grid_info_2d.dbu), int(-cd/2/grid_info_2d.dbu)], 
        [int(-cd/2/grid_info_2d.dbu), int(cd/2/grid_info_2d.dbu)], 
        [int(cd/2/grid_info_2d.dbu), int(cd/2/grid_info_2d.dbu)], 
        [int(cd/2/grid_info_2d.dbu), int(-cd/2/grid_info_2d.dbu)]
    ]
])
g2 = geo_manager([[
        [int(-pitch/2/grid_info_2d.dbu), int(-pitch/8/grid_info_2d.dbu)], 
        [int(-pitch/2/grid_info_2d.dbu), int(pitch/8/grid_info_2d.dbu)], 
        [int(-pitch/4/grid_info_2d.dbu), int(pitch/8/grid_info_2d.dbu)], 
        [int(-pitch/4/grid_info_2d.dbu), int(-pitch/8/grid_info_2d.dbu)]
    ],[
        [int(pitch/4/grid_info_2d.dbu), int(-pitch/2/grid_info_2d.dbu)], 
        [int(pitch/4/grid_info_2d.dbu), int(pitch/2/grid_info_2d.dbu)], 
        [int(pitch/2/grid_info_2d.dbu), int(pitch/2/grid_info_2d.dbu)], 
        [int(pitch/2/grid_info_2d.dbu), int(-pitch/2/grid_info_2d.dbu)]
    ],

])
mask = binary_mask.create(absorber, background, grid_info_2d, g2.get_vertex())
grid_info_2d.display(mask)
complex_data_array = np.array(mask, dtype=np.complex64).reshape((size, size))
print(f"near field error = {np.max(complex_data_array - near_field)}")

In [ ]:
fourier_shifted = np.fft.fftshift(np.fft.fft2(complex_data_array))

difract = diffraction(grid_info_2d)
print(difract.update_diffraction_source_points(fourier_shifted.flatten().tolist()))
pupil_intensity = difract.get_imaging_pupil_intensity([100, 100])

amp_error = [a - x for a, x in zip(pupil.flatten().tolist(), pupil_intensity)]
assert(max(np.abs(amp_error)) < grid_info_2d.dbu)

In [ ]:
def shift_phase(ix, iy, N, delta_x=0.5, delta_y=0.5):
    return np.exp(-2j * np.pi * (ix * delta_x / N + iy * delta_y / N))

desired_orders = get_diffraction_order(grid_info_2d)
print(f"保留的衍射级次: {desired_orders}")

# TE = np.zeros_like(fourier_shifted)
TE = np.zeros((40, 40), dtype=np.complex64)
TM = np.zeros_like(TE)

In [91]:

reconstructed_data_TE_1 = np.loadtxt(os.path.join(os.path.abspath(os.path.expanduser(golden_root_dir)), "thin_mask_2d_aerial_binary_TE.txt"), delimiter='\t')

reconstructed_data_TM_1 = np.loadtxt(os.path.join(os.path.abspath(os.path.expanduser(golden_root_dir)), "thin_mask_2d_aerial_binary_TM.txt"), delimiter='\t')

reconstructed_data_magnitude = (reconstructed_data_TE + reconstructed_data_TM) / 2


In [92]:
def plotnx1(im_list, name_list, n = 4):
    fig, axes = plt.subplots(1, n, figsize=(12, 6))
    def sub_plot(n, data, name):
        fig.colorbar(axes[n].imshow(data, cmap='viridis'), ax=axes[n], label='')
        axes[n].set_title(name)
        axes[n].axis('off')
    for i in range(n):
        sub_plot(i, im_list[i], name_list[i])
    plt.tight_layout() 
    plt.show()
    print(f"max {name_list[-1]}={np.max(im_list[-1])}")
def normlization(golden, input):
    return input * np.max(golden)/np.max(input)
def normlization_diff(golden, input):
    return np.abs(golden - normlization(golden, input)) / 1

In [ ]:
plotnx1([reconstructed_data_TE, reconstructed_data_TM, normlization_diff(reconstructed_data_TE, reconstructed_data_TM)], 
        ["TE from simulation", "TM from simulation", "diff"], 3
)